In [1]:
import warnings
warnings.filterwarnings('ignore')
import pyspark
from pyspark.ml.feature import StringIndexer

In [2]:
spark = (
    pyspark.sql.SparkSession.builder
        .appName("Spark ML Research")
        .master("local[4]") # limit executor to 4 cores
        .config("spark.executor.memory", "1g")
        .config("spark.driver.memory", "1g")
        .getOrCreate()
)
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)  # to pretty print pyspark.DataFrame in jupyter

sc = spark.sparkContext
sc.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/05 15:48:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.createDataFrame([('a', 1.0), ('a', 2.0), (float("nan"), 3.0),
                            ('b', 4.0), ('c', 5.0)], ["a_col", "b_col"])
df

a_col,b_col
a,1.0
a,2.0
NaN,3.0
b,4.0
c,5.0


`StringIndexer` преобразует категориальные строки/числа в индексы 0,1,2,... для последующего ML

In [4]:
indexer = StringIndexer()\
    .setInputCol("a_col")\
    .setOutputCol("aIndex")
indexer.fit(df).transform(df).show()

+-----+-----+------+
|a_col|b_col|aIndex|
+-----+-----+------+
|    a|  1.0|   0.0|
|    a|  2.0|   0.0|
|  NaN|  3.0|   1.0|
|    b|  4.0|   2.0|
|    c|  5.0|   3.0|
+-----+-----+------+



In [5]:
df = spark.createDataFrame([('a', 1.0), ('a', 2.0), (None, 3.0),
                            ('b', 4.0), ('c', 5.0)], ["a", "b"])
df

a,b
a,1.0
a,2.0
NULL,3.0
b,4.0
c,5.0


In [6]:
indexer = StringIndexer()\
    .setInputCol("a")\
    .setOutputCol("aIndex")
indexed = indexer.fit(df).transform(df)

In [7]:
indexed.show(1)

25/09/05 15:48:09 ERROR Executor: Exception in task 1.0 in stage 23.0 (TID 34)
org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] User defined function (`StringIndexerModel$$Lambda$4042/0x000000e002112a58`: (string) => double) failed due to: org.apache.spark.SparkException: StringIndexer encountered NULL value. To handle or skip NULLS, try setting StringIndexer.handleInvalid.. SQLSTATE: 39000
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:195)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvalu

Py4JJavaError: An error occurred while calling o179.showString.
: org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] User defined function (`StringIndexerModel$$Lambda$4042/0x000000e002112a58`: (string) => double) failed due to: org.apache.spark.SparkException: StringIndexer encountered NULL value. To handle or skip NULLS, try setting StringIndexer.handleInvalid.. SQLSTATE: 39000
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:195)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.s

## Ручная чистка

In [8]:
df = df.fillna('', subset=['a'])
df

a,b
a,1.0
a,2.0
,3.0
b,4.0
c,5.0


In [9]:
indexer = StringIndexer()\
    .setInputCol("a")\
    .setOutputCol("aIndex")
indexed = indexer.fit(df).transform(df)
indexed.show()

+---+---+------+
|  a|  b|aIndex|
+---+---+------+
|  a|1.0|   0.0|
|  a|2.0|   0.0|
|   |3.0|   1.0|
|  b|4.0|   2.0|
|  c|5.0|   3.0|
+---+---+------+



## Авто обработка

In [10]:
df = spark.createDataFrame([('a', 1.0), ('a', 2.0), (None, 3.0),
                            ('b', 4.0), ('c', 5.0)], ["a", "b"])
df

a,b
a,1.0
a,2.0
NULL,3.0
b,4.0
c,5.0


In [11]:
indexer = StringIndexer(handleInvalid='keep')\
    .setInputCol("a")\
    .setOutputCol("aIndex")
indexed = indexer.fit(df).transform(df)
indexed.show()

+----+---+------+
|   a|  b|aIndex|
+----+---+------+
|   a|1.0|   0.0|
|   a|2.0|   0.0|
|NULL|3.0|   3.0|
|   b|4.0|   1.0|
|   c|5.0|   2.0|
+----+---+------+



In [12]:
spark.stop()